In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import module to interact with database collection
from CRUD_Python_Module import CollectionManager

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "password"

# Connect to database via CRUD Module
db = CollectionManager(username=username, pwd=password,
                       db='aac', col='animals',
                           host='localhost', port=27017)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={'width': '150px', 'height': '150px'}),
    html.H4("Developed by Jenna Gorham on a cloudy Monday"),
    html.Hr(),
    # Interactive filtering
    html.Div([html.Button(id='reset-button', children='Reset All', n_clicks=0)]),
    dcc.Checklist(id='filter-checklist',
                  options=[
                      {'label': 'Water Rescue', 'value': 'water'},
                      {'label': 'Mountain / Wilderness Rescue', 'value': 'wilderness'},
                      {'label': 'Disaster Rescue / Individual Tracking', 'value': 'disaster'}
                  ],
                  inline=True
                 ),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
        ],
        data=df.to_dict('records'),
        editable=False,
        filter_action='native',
        sort_action='native',
        sort_mode='multi',
        column_selectable=False,
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action='native',
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'scroll', 'width': '100%'}
    ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ]),
    html.Br(),
    html.Hr(),
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'width': '150px', 'height': '150px'}),
    html.H4("Developed by Jenna Gorham on a cloudy Monday")
])

#############################################
# Interaction Between Components / Controller
#############################################


# Reset all filters
@app.callback(Output('filter-checklist', 'value'),
              Input('reset-button', 'n_clicks'))
def reset_dashboard(button_clicks):
    return []


# Update dashboard table based on filter selections
# Allows for multiple selections to be displayed simultaneously
# If no filters are selected, displays all data
@app.callback(Output('datatable-id','data'),
              [Input('filter-checklist', 'value')])
def update_dashboard(selected_filters):
    # Define queries used for each filter option
    queries = {'water': 
               {'$and': [
                   {'animal_type': 'Dog'},
                   {'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']}},
                   {'sex_upon_outcome': 'Intact Female'},
                   {'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
               ]},
               'wilderness': 
               {'$and': [
                   {'animal_type': 'Dog'},
                   {'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                                      'Siberian Husky', 'Rottweiler']}},
                   {'sex_upon_outcome': 'Intact Male'},
                   {'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
               ]},
               'disaster':
               {'$and': [
                   {'animal_type': 'Dog'},
                   {'breed': {'$in': ['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                                      'Bloodhound', 'Rottweiler']}},
                   {'sex_upon_outcome': 'Intact Male'},
                   {'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
               ]}
              }
    
    # Read from db based on selections
    # In case of multiple selections, combine results of multiple queries using '$or'
    if selected_filters:
        # List comprehension to build a query combining multiple conditions dynamically
        query = {'$or': [queries[db_filter] for db_filter in selected_filters]}
    else:
        query = {}
    df = pd.DataFrame.from_records(db.read(query))
    df.drop(columns=['_id'], inplace=True)
    return df.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        return
    dff = pd.DataFrame.from_dict(viewData)
    mask = dff['animal_type'] == 'Dog'
    dff_agg = dff[mask].groupby('breed')['rec_num'].agg('count').reset_index().sort_values(by='rec_num', ascending=False)
    fig = px.bar(dff_agg, x='breed', y='rec_num',
                labels={'rec_num': 'count'},
                title='Breeds of Selected Dogs')
    return dcc.Graph(figure=fig)

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '700px', 'height': '450px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Popup([
                    html.H1("Animal Info"),
                    html.P(f"Name:\t {dff.iloc[row,9]}"),
                    html.P(f"Breed:\t {dff.iloc[row]['breed']}"),
                    html.P(f"Sex:\t {dff.iloc[row]['sex_upon_outcome']}"),
                    html.P(f"Age (weeks):\t {round(dff.iloc[row]['age_upon_outcome_in_weeks'], 1)}")
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://escaperobot-subjectcuba-3000.codio.io/proxy/8050/
